#### npz File MFCC

In [ ]:
import os
import librosa
import numpy as np

ulazni_dir = "datasetSpeech_1"
fajlovi = [f for f in os.listdir(ulazni_dir) if f.endswith(".wav")]

# treci broj u imenu fajla
EMOTION_MAP = {
    "01": 0,  # neutral
    "02": 1,  # calm
    "03": 2,  # happy
    "04": 3,  # sad
    "05": 4,  # angry
    "06": 5,  # fearful
    "07": 6,  # disgust
    "08": 7,  # surprised
}

X = []
y = []
speakers = []  # NOVO: speaker ID-evi
filenames = []  # NOVO: originalni filenames

target_sr = 16000
n_mfcc = 40 

print(f"MFCC ekstrakcija {len(fajlovi)} fajlova sa {n_mfcc} koeficijenata")

for i, fajl in enumerate(fajlovi):
    try:
        deli = fajl.split("-")
        emocija_kod = deli[2]
        labela = EMOTION_MAP[emocija_kod]
        
        # id
        speaker_str = deli[6].split(".")[0]
        speaker_id = int(speaker_str)

        putanja = os.path.join(ulazni_dir, fajl)
        audio, sr = librosa.load(putanja, sr=target_sr, mono=True)

        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)

        X.append(mfcc)
        y.append(labela)
        speakers.append(speaker_id)  # NOVO
        filenames.append(fajl)        # NOVO

    except Exception as e:
        print(f"greska na fajlu {fajl}: {e}")

    if (i + 1) % 200 == 0 or (i + 1) == len(fajlovi):
        print(f"obradjeno {i + 1}/{len(fajlovi)}")


X = np.array(X)
y = np.array(y)
speakers = np.array(speakers, dtype=np.int32)  
filenames = np.array(filenames, dtype=object)  
X = np.expand_dims(X, axis=-1)

print("\nKRAJ")
print(f"Oblik ulaznih podataka (X): {X.shape}") 
print(f"Oblik labela (y):           {y.shape}")
print(f"Oblik speaker ID-eva:       {speakers.shape}")  

np.savez_compressed(
    "dataset_mfcc.npz", 
    X=X, 
    y=y,
    speakers=speakers,
    filenames=filenames
)
print("Sačuvano u 'dataset_mfcc.npz' sa speakers i filenames!")

MFCC ekstrakcija 1440 fajlova sa 40 koeficijenata
obradjeno 200/1440
obradjeno 400/1440
obradjeno 600/1440
obradjeno 800/1440
obradjeno 1000/1440
obradjeno 1200/1440
obradjeno 1400/1440
obradjeno 1440/1440

KRAJ
Oblik ulaznih podataka (X): (1440, 40, 110, 1)
Oblik labela (y):           (1440,)
Oblik speaker ID-eva:       (1440,)
Sačuvano u 'dataset_mfcc.npz' sa speakers i filenames!


In [ ]:
import numpy as np
import os
import torch
from torch.utils.data import Dataset, DataLoader

def spec_augment_mfcc_v2(
    mfcc_spec, W=3, freq_mask_max=2, time_mask_max=5, num_masks=1
):
    augmented = mfcc_spec.copy()
    is_3d = augmented.ndim == 3 and augmented.shape[-1] == 1
    if is_3d:
        augmented = augmented[:, :, 0]

    num_freqs, num_steps = augmented.shape

    if num_steps > 2 * W + 1 and W > 0 and np.random.random() < 0.2:
        center = np.random.randint(W, num_steps - W)
        warped_center = center + np.random.randint(-W, W + 1)

        orig_points = np.array([0, center, num_steps - 1])
        warped_points = np.array([0, warped_center, num_steps - 1])

        new_time = np.arange(num_steps)
        src_time = np.interp(new_time, warped_points, orig_points)

        warped_spec = np.zeros_like(augmented)
        for f_idx in range(num_freqs):
            warped_spec[f_idx, :] = np.interp(
                src_time, np.arange(num_steps), augmented[f_idx, :]
            )
        augmented = warped_spec

    if np.random.random() < 0.8:
        f = np.random.randint(1, freq_mask_max)
        f0 = np.random.randint(0, max(1, num_freqs - f))
        augmented[f0 : f0 + f, :] = 0

    if np.random.random() < 0.8:
        t = np.random.randint(1, time_mask_max)
        t0 = np.random.randint(0, max(1, num_steps - t))
        augmented[:, t0 : t0 + t] = 0

    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)

    return augmented


# load
print("Učitavanje dataset_mfcc.npz...")
data = np.load("dataset_mfcc.npz", allow_pickle=True)
X_all = data["X"]
y_all = data["y"]
speakers_all = data["speakers"]      
filenames_all = data["filenames"]    

print(f"✓ Učitano: {X_all.shape[0]} uzoraka")
print(f"✓ Shape X: {X_all.shape}")
print(f"✓ Speakers: min={speakers_all.min()}, max={speakers_all.max()}")


unique_speakers = np.unique(speakers_all)
for speaker in sorted(unique_speakers):
    count = np.sum(speakers_all == speaker)
    if 1 <= speaker <= 20:
        split = "TRAIN"
    elif speaker in [21, 22]:
        split = "VAL"
    elif speaker in [23, 24]:
        split = "TEST"
    else:
        split = "UNKNOWN"
    print(f"Speaker {speaker:2d} ({split:6s}): {count:3d} uzoraka")

# speaker independent split
X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []


train_speakers = set(range(1, 21))      # 1-20 actor
val_speakers = set([21, 22])            # 21-22 actor
test_speakers = set([23, 24])           # 23-24 actor



for c in range(8):
    indices = np.where(y_all == c)[0]
    
    
    train_indices = indices[np.isin(speakers_all[indices], list(train_speakers))]
    val_indices = indices[np.isin(speakers_all[indices], list(val_speakers))]
    test_indices = indices[np.isin(speakers_all[indices], list(test_speakers))]
    
    
    train_w = []
    for idx in train_indices:
        fname = str(filenames_all[idx])
        parts = fname.split("-")
        if len(parts) >= 4 and parts[3] == "02":
            train_w.append(2.0)
        else:
            train_w.append(1.0)
    train_w = np.array(train_w, dtype=np.float32)
    
    val_w = []
    for idx in val_indices:
        fname = str(filenames_all[idx])
        parts = fname.split("-")
        if len(parts) >= 4 and parts[3] == "02":
            val_w.append(2.0)
        else:
            val_w.append(1.0)
    val_w = np.array(val_w, dtype=np.float32)
    
    test_w = []
    for idx in test_indices:
        fname = str(filenames_all[idx])
        parts = fname.split("-")
        if len(parts) >= 4 and parts[3] == "02":
            test_w.append(2.0)
        else:
            test_w.append(1.0)
    test_w = np.array(test_w, dtype=np.float32)
    
    X_orig = X_all[train_indices]
    
    if c == 0:
        X_aug = np.array([spec_augment_mfcc_v2(x, W=2, freq_mask_max=2, 
                                                time_mask_max=5, num_masks=1) 
                         for x in X_orig])
        
        X_tr.append(np.concatenate((X_orig, X_aug), axis=0))
        y_tr.append(np.tile(y_all[train_indices], 2))
        w_tr.append(np.tile(train_w, 2))
        
        print(f"Klasa {c} (NEUTRAL): {len(train_indices)} train → {len(train_indices) * 2} sa aug")
    else:
        X_tr.append(X_orig)
        y_tr.append(y_all[train_indices])
        w_tr.append(train_w)
        
        print(f"Klasa {c}: {len(train_indices)} train uzoraka")
    
    if len(val_indices) > 0:
        X_va.append(X_all[val_indices])
        y_va.append(y_all[val_indices])
        w_va.append(val_w)
        print(f"  → Val: {len(val_indices)} uzoraka")
    
    if len(test_indices) > 0:
        X_te.append(X_all[test_indices])
        y_te.append(y_all[test_indices])
        w_te.append(test_w)
        print(f"  → Test: {len(test_indices)} uzoraka")


X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]

print(f"\n=== FINALNA SPEAKER-INDEPENDENT PODELA ===")
print(f"Train: {X_train.shape[0]} uzoraka (govorinici 1-20)")
print(f"Val:   {X_val.shape[0]} uzoraka (govorinici 21-22)")
print(f"Test:  {X_test.shape[0]} uzoraka (govorinici 23-24)")



mean = np.mean(X_train, axis=(0, 2, 3), keepdims=True)  # (1, 40, 1, 1)
std = np.std(X_train, axis=(0, 2, 3), keepdims=True)    # (1, 40, 1, 1)

print(f"Mean shape: {mean.shape}, Std shape: {std.shape}")

X_train = (X_train - mean) / (std + 1e-8)

X_val = (X_val - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)



# ============================================
# 6. PYTORCH DATASET
# ============================================
class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_mfcc = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_mfcc = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_mfcc = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)

print(f"\n✓ Data loaderi kreirani!")
print(f"  Train loader: {len(train_loader_mfcc)} batcheva")
print(f"  Val loader: {len(val_loader_mfcc)} batcheva")
print(f"  Test loader: {len(test_loader_mfcc)} batcheva")

Učitavanje dataset_mfcc.npz...
✓ Učitano: 1440 uzoraka
✓ Shape X: (1440, 40, 110, 1)
✓ Speakers: min=1, max=24

=== SPEAKER STATISTIKA ===
Speaker  1 (TRAIN ):  60 uzoraka
Speaker  2 (TRAIN ):  60 uzoraka
Speaker  3 (TRAIN ):  60 uzoraka
Speaker  4 (TRAIN ):  60 uzoraka
Speaker  5 (TRAIN ):  60 uzoraka
Speaker  6 (TRAIN ):  60 uzoraka
Speaker  7 (TRAIN ):  60 uzoraka
Speaker  8 (TRAIN ):  60 uzoraka
Speaker  9 (TRAIN ):  60 uzoraka
Speaker 10 (TRAIN ):  60 uzoraka
Speaker 11 (TRAIN ):  60 uzoraka
Speaker 12 (TRAIN ):  60 uzoraka
Speaker 13 (TRAIN ):  60 uzoraka
Speaker 14 (TRAIN ):  60 uzoraka
Speaker 15 (TRAIN ):  60 uzoraka
Speaker 16 (TRAIN ):  60 uzoraka
Speaker 17 (TRAIN ):  60 uzoraka
Speaker 18 (TRAIN ):  60 uzoraka
Speaker 19 (TRAIN ):  60 uzoraka
Speaker 20 (TRAIN ):  60 uzoraka
Speaker 21 (VAL   ):  60 uzoraka
Speaker 22 (VAL   ):  60 uzoraka
Speaker 23 (TEST  ):  60 uzoraka
Speaker 24 (TEST  ):  60 uzoraka

=== PODELA PO KLASAMA ===
Klasa 0 (NEUTRAL): 80 train → 160 sa aug
 

#### 2N

In [17]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc2N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_mfcc_2N_best.pth")


class MFCC_2CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_2CNN_Small, self).__init__()
        n = 2
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

#m METRIKE 2N

def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = MFCC_2CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}


# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), best_model_path)
        saved_flag = " [Model je sačuvan]"

    
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_mfcc:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)

        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))


# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

#finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))




Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.9920 | Val Loss: 2.0731 | Val Acc: 10.83% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.9757 | Val Loss: 2.0667 | Val Acc: 14.17% [Model je sačuvan]
Epoha 03/30 | Train Loss: 2.9666 | Val Loss: 2.0614 | Val Acc: 23.33% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.9603 | Val Loss: 2.0563 | Val Acc: 22.50% [Model je sačuvan]
Epoha 05/30 | Train Loss: 2.9541 | Val Loss: 2.0530 | Val Acc: 18.33% [Model je sačuvan]
Epoha 06/30 | Train Loss: 2.9407 | Val Loss: 2.0483 | Val Acc: 16.67% [Model je sačuvan]
Epoha 07/30 | Train Loss: 2.9275 | Val Loss: 2.0443 | Val Acc: 17.50% [Model je sačuvan]
Epoha 08/30 | Train Loss: 2.9275 | Val Loss: 2.0400 | Val Acc: 19.17% [Model je sačuvan]
Epoha 09/30 | Train Loss: 2.9159 | Val Loss: 2.0372 | Val Acc: 18.33% [Model je sačuvan]
Epoha 10/30 | Train Loss: 2.8977 | Val Loss: 2.0328 | Val Acc: 19.17% [Model je sačuvan]
Epoha 11/30 | Train Loss: 2.8906 | Val Loss: 2.0317 | Val Acc: 20.00% [Model je sačuvan

In [18]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_2CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_2N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

=== Predikcija za 8. fajl u test skupu ===
Stvarna emocija (Ground Truth): fearful
Predviđena emocija:             calm (20.62%)
Vreme pojedinačne inferencije:  12.444 ms

Verovatnoće po klasama:
  neutral   :  17.22%
  calm      :  20.62%
  happy     :   8.09%
  sad       :  15.71%
  angry     :   4.84%
  fearful   :   8.08%
  disgust   :  12.10%
  surprised :  13.34%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           0.1861 s
Prosečno vreme po uzorku:  1.551 ms
Brzina obrade (throughput): 644.73 FPS (uzoraka/s)


#### 4N

In [8]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc4N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_mfcc_4N_best.pth")


class MFCC_4CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_4CNN_Small, self).__init__()
        n = 4
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

#m METRIKE 4N

def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = MFCC_4CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}


# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), best_model_path)
        saved_flag = " [Model je sačuvan]"

    
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_mfcc:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)

        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))


# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

#finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))




Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 3.0277 | Val Loss: 2.0757 | Val Acc: 13.33% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.9797 | Val Loss: 2.0625 | Val Acc: 13.33% [Model je sačuvan]
Epoha 03/30 | Train Loss: 2.9578 | Val Loss: 2.0538 | Val Acc: 12.50% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.9436 | Val Loss: 2.0442 | Val Acc: 12.50% [Model je sačuvan]
Epoha 05/30 | Train Loss: 2.9239 | Val Loss: 2.0334 | Val Acc: 14.17% [Model je sačuvan]
Epoha 06/30 | Train Loss: 2.9003 | Val Loss: 2.0229 | Val Acc: 15.00% [Model je sačuvan]
Epoha 07/30 | Train Loss: 2.8787 | Val Loss: 2.0108 | Val Acc: 17.50% [Model je sačuvan]
Epoha 08/30 | Train Loss: 2.8375 | Val Loss: 1.9863 | Val Acc: 23.33% [Model je sačuvan]
Epoha 09/30 | Train Loss: 2.8053 | Val Loss: 1.9676 | Val Acc: 21.67% [Model je sačuvan]
Epoha 10/30 | Train Loss: 2.7674 | Val Loss: 1.9436 | Val Acc: 25.83% [Model je sačuvan]
Epoha 11/30 | Train Loss: 2.7187 | Val Loss: 1.9203 | Val Acc: 25.83% [Model je sačuvan

In [9]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_4CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_4N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

=== Predikcija za 8. fajl u test skupu ===
Stvarna emocija (Ground Truth): fearful
Predviđena emocija:             neutral (30.95%)
Vreme pojedinačne inferencije:  3.218 ms

Verovatnoće po klasama:
  neutral   :  30.95%
  calm      :   2.84%
  happy     :  15.60%
  sad       :   6.77%
  angry     :   5.30%
  fearful   :  13.42%
  disgust   :   5.71%
  surprised :  19.41%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           0.0432 s
Prosečno vreme po uzorku:  0.360 ms
Brzina obrade (throughput): 2775.66 FPS (uzoraka/s)


#### 8N

In [13]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc8N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_mfcc_8N_best.pth")


class MFCC_8CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_8CNN_Small, self).__init__()
        n = 8
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

#m METRIKE 8n

def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = MFCC_8CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}


# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), best_model_path)
        saved_flag = " [Model je sačuvan]"

    
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_mfcc:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)

        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))


# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

#finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))




Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.9761 | Val Loss: 2.0533 | Val Acc: 10.83% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.9371 | Val Loss: 2.0383 | Val Acc: 14.17% [Model je sačuvan]
Epoha 03/30 | Train Loss: 2.9124 | Val Loss: 2.0296 | Val Acc: 15.83% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.8912 | Val Loss: 2.0205 | Val Acc: 14.17% [Model je sačuvan]
Epoha 05/30 | Train Loss: 2.8583 | Val Loss: 2.0074 | Val Acc: 13.33% [Model je sačuvan]
Epoha 06/30 | Train Loss: 2.8266 | Val Loss: 1.9915 | Val Acc: 21.67% [Model je sačuvan]
Epoha 07/30 | Train Loss: 2.7891 | Val Loss: 1.9750 | Val Acc: 29.17% [Model je sačuvan]
Epoha 08/30 | Train Loss: 2.7494 | Val Loss: 1.9498 | Val Acc: 30.00% [Model je sačuvan]
Epoha 09/30 | Train Loss: 2.7115 | Val Loss: 1.9324 | Val Acc: 29.17% [Model je sačuvan]
Epoha 10/30 | Train Loss: 2.6724 | Val Loss: 1.8972 | Val Acc: 35.00% [Model je sačuvan]
Epoha 11/30 | Train Loss: 2.6210 | Val Loss: 1.8783 | Val Acc: 35.83% [Model je sačuvan

In [14]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_8CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_8N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

=== Predikcija za 8. fajl u test skupu ===
Stvarna emocija (Ground Truth): fearful
Predviđena emocija:             happy (19.21%)
Vreme pojedinačne inferencije:  10.862 ms

Verovatnoće po klasama:
  neutral   :  18.43%
  calm      :   3.64%
  happy     :  19.21%
  sad       :   9.26%
  angry     :   8.50%
  fearful   :  17.67%
  disgust   :   7.18%
  surprised :  16.10%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           0.1678 s
Prosečno vreme po uzorku:  1.398 ms
Brzina obrade (throughput): 715.12 FPS (uzoraka/s)


#### 16N

In [15]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc16N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_mfcc_16N_best.pth")


class MFCC_16CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_16CNN_Small, self).__init__()
        n = 16
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

#m METRIKE 16n

def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = MFCC_16CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}


# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), best_model_path)
        saved_flag = " [Model je sačuvan]"

    
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_mfcc:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)

        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))


# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

#finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))




Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.9361 | Val Loss: 2.0240 | Val Acc: 20.83% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.8667 | Val Loss: 2.0026 | Val Acc: 19.17% [Model je sačuvan]
Epoha 03/30 | Train Loss: 2.8028 | Val Loss: 1.9665 | Val Acc: 23.33% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.7117 | Val Loss: 1.9065 | Val Acc: 33.33% [Model je sačuvan]
Epoha 05/30 | Train Loss: 2.6265 | Val Loss: 2.0549 | Val Acc: 20.00%
Epoha 06/30 | Train Loss: 2.5303 | Val Loss: 1.8479 | Val Acc: 32.50% [Model je sačuvan]
Epoha 07/30 | Train Loss: 2.4387 | Val Loss: 1.8756 | Val Acc: 34.17%
Epoha 08/30 | Train Loss: 2.3541 | Val Loss: 1.6905 | Val Acc: 34.17% [Model je sačuvan]
Epoha 09/30 | Train Loss: 2.2566 | Val Loss: 1.7634 | Val Acc: 35.00%
Epoha 10/30 | Train Loss: 2.1956 | Val Loss: 1.8683 | Val Acc: 35.00%
Epoha 11/30 | Train Loss: 2.1123 | Val Loss: 1.6247 | Val Acc: 40.00% [Model je sačuvan]
Epoha 12/30 | Train Loss: 2.0576 | Val Loss: 1.5421 | Val Acc: 45.00% [Mod

In [16]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_16CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_16N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

=== Predikcija za 8. fajl u test skupu ===
Stvarna emocija (Ground Truth): fearful
Predviđena emocija:             disgust (22.18%)
Vreme pojedinačne inferencije:  13.804 ms

Verovatnoće po klasama:
  neutral   :  11.90%
  calm      :   6.49%
  happy     :  16.46%
  sad       :  11.31%
  angry     :   9.84%
  fearful   :  12.44%
  disgust   :  22.18%
  surprised :   9.38%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           0.3681 s
Prosečno vreme po uzorku:  3.067 ms
Brzina obrade (throughput): 326.02 FPS (uzoraka/s)


#### 32N


In [19]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc32N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_mfcc_32N_best.pth")


class MFCC_32CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_32CNN_Small, self).__init__()
        n = 32
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

#m METRIKE 32N

def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = MFCC_32CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}


# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), best_model_path)
        saved_flag = " [Model je sačuvan]"

    
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_mfcc:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)

        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))


# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

#finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))




Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.8968 | Val Loss: 1.9992 | Val Acc: 20.83% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.7449 | Val Loss: 1.9166 | Val Acc: 24.17% [Model je sačuvan]
Epoha 03/30 | Train Loss: 2.5606 | Val Loss: 1.8097 | Val Acc: 35.83% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.3816 | Val Loss: 2.1949 | Val Acc: 18.33%
Epoha 05/30 | Train Loss: 2.2353 | Val Loss: 1.6500 | Val Acc: 44.17% [Model je sačuvan]
Epoha 06/30 | Train Loss: 2.1296 | Val Loss: 1.9995 | Val Acc: 25.00%
Epoha 07/30 | Train Loss: 2.0262 | Val Loss: 1.4595 | Val Acc: 44.17% [Model je sačuvan]
Epoha 08/30 | Train Loss: 1.9412 | Val Loss: 1.6424 | Val Acc: 35.83%
Epoha 09/30 | Train Loss: 1.8796 | Val Loss: 1.7895 | Val Acc: 32.50%
Epoha 10/30 | Train Loss: 1.7956 | Val Loss: 1.6070 | Val Acc: 35.00%
Epoha 11/30 | Train Loss: 1.7324 | Val Loss: 1.7500 | Val Acc: 41.67%
Epoha 12/30 | Train Loss: 1.6687 | Val Loss: 1.3367 | Val Acc: 50.00% [Model je sačuvan]
Epoha 13/30 | Train Los

In [20]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_32CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_32N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

=== Predikcija za 8. fajl u test skupu ===
Stvarna emocija (Ground Truth): fearful
Predviđena emocija:             surprised (42.97%)
Vreme pojedinačne inferencije:  69.470 ms

Verovatnoće po klasama:
  neutral   :   0.67%
  calm      :   0.13%
  happy     :  21.22%
  sad       :   3.56%
  angry     :  12.27%
  fearful   :  11.06%
  disgust   :   8.12%
  surprised :  42.97%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           0.3214 s
Prosečno vreme po uzorku:  2.679 ms
Brzina obrade (throughput): 373.33 FPS (uzoraka/s)


#### 64N

In [35]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc64N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_mfcc_64N_best.pth")


class MFCC_64CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_64CNN_Small, self).__init__()
        n = 64
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

#m METRIKE 2N

def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = MFCC_64CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}


# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), best_model_path)
        saved_flag = " [Model je sačuvan]"

    
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_mfcc:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)

        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))


# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

#finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))




Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.8588 | Val Loss: 1.9658 | Val Acc: 20.83% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.5772 | Val Loss: 1.9380 | Val Acc: 22.50% [Model je sačuvan]
Epoha 03/30 | Train Loss: 2.3061 | Val Loss: 1.7389 | Val Acc: 35.00% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.1041 | Val Loss: 1.9274 | Val Acc: 23.33%
Epoha 05/30 | Train Loss: 1.9563 | Val Loss: 2.1797 | Val Acc: 26.67%
Epoha 06/30 | Train Loss: 1.8998 | Val Loss: 1.4122 | Val Acc: 45.00% [Model je sačuvan]
Epoha 07/30 | Train Loss: 1.7809 | Val Loss: 1.8352 | Val Acc: 24.17%
Epoha 08/30 | Train Loss: 1.7345 | Val Loss: 1.6871 | Val Acc: 38.33%
Epoha 09/30 | Train Loss: 1.6000 | Val Loss: 1.5919 | Val Acc: 45.83%
Epoha 10/30 | Train Loss: 1.6040 | Val Loss: 1.9510 | Val Acc: 33.33%
Epoha 11/30 | Train Loss: 1.4541 | Val Loss: 1.7745 | Val Acc: 33.33%
Epoha 12/30 | Train Loss: 1.3591 | Val Loss: 1.8433 | Val Acc: 36.67%
Epoha 13/30 | Train Loss: 1.3072 | Val Loss: 2.0049 | Val Acc

In [36]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_64CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_64N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

=== Predikcija za 67. fajl u test skupu ===
Stvarna emocija (Ground Truth): sad
Predviđena emocija:             calm (46.17%)
Vreme pojedinačne inferencije:  50.700 ms

Verovatnoće po klasama:
  neutral   :  43.37%
  calm      :  46.17%
  happy     :   0.51%
  sad       :   8.90%
  angry     :   0.00%
  fearful   :   0.72%
  disgust   :   0.32%
  surprised :   0.01%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           1.1050 s
Prosečno vreme po uzorku:  9.209 ms
Brzina obrade (throughput): 108.60 FPS (uzoraka/s)


#### 128N

In [24]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc128N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_mfcc_128N_best.pth")


class MFCC_128CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_128CNN_Small, self).__init__()
        n = 128
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

#m METRIKE 128N

def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = MFCC_128CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}


# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), best_model_path)
        saved_flag = " [Model je sačuvan]"

    
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_mfcc:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)

        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))


# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

#finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))




Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.7924 | Val Loss: 1.9269 | Val Acc: 23.33% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.4015 | Val Loss: 1.8712 | Val Acc: 26.67% [Model je sačuvan]
Epoha 03/30 | Train Loss: 2.1362 | Val Loss: 1.7563 | Val Acc: 35.00% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.0221 | Val Loss: 1.5053 | Val Acc: 44.17% [Model je sačuvan]
Epoha 05/30 | Train Loss: 1.8775 | Val Loss: 2.2227 | Val Acc: 23.33%
Epoha 06/30 | Train Loss: 1.8061 | Val Loss: 1.8858 | Val Acc: 39.17%
Epoha 07/30 | Train Loss: 1.7248 | Val Loss: 1.7297 | Val Acc: 28.33%
Epoha 08/30 | Train Loss: 1.6566 | Val Loss: 2.5309 | Val Acc: 31.67%
Epoha 09/30 | Train Loss: 1.4353 | Val Loss: 2.2026 | Val Acc: 35.83%
Epoha 10/30 | Train Loss: 1.3642 | Val Loss: 3.1957 | Val Acc: 18.33%
Epoha 11/30 | Train Loss: 1.2614 | Val Loss: 2.4672 | Val Acc: 26.67%
Epoha 12/30 | Train Loss: 1.2436 | Val Loss: 1.9703 | Val Acc: 44.17%
Epoha 13/30 | Train Loss: 1.1305 | Val Loss: 1.3813 | Val Acc

In [25]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_128CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_128N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

=== Predikcija za 8. fajl u test skupu ===
Stvarna emocija (Ground Truth): fearful
Predviđena emocija:             neutral (20.35%)
Vreme pojedinačne inferencije:  25.935 ms

Verovatnoće po klasama:
  neutral   :  20.35%
  calm      :   6.17%
  happy     :  16.22%
  sad       :   7.10%
  angry     :   7.64%
  fearful   :  16.96%
  disgust   :   5.95%
  surprised :  19.62%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           1.6875 s
Prosečno vreme po uzorku:  14.063 ms
Brzina obrade (throughput): 71.11 FPS (uzoraka/s)


#### 256N

In [26]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc256N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_mfcc_256N_best.pth")


class MFCC_256CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_256CNN_Small, self).__init__()
        n = 256
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

#m METRIKE 256N

def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = MFCC_256CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}


# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), best_model_path)
        saved_flag = " [Model je sačuvan]"

    
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_mfcc:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)

        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))


# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

#finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))




Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.8055 | Val Loss: 2.2274 | Val Acc: 20.83% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.3582 | Val Loss: 2.4021 | Val Acc: 19.17%
Epoha 03/30 | Train Loss: 2.0990 | Val Loss: 1.7403 | Val Acc: 33.33% [Model je sačuvan]
Epoha 04/30 | Train Loss: 1.9681 | Val Loss: 1.8403 | Val Acc: 30.00%
Epoha 05/30 | Train Loss: 1.9006 | Val Loss: 1.4592 | Val Acc: 50.83% [Model je sačuvan]
Epoha 06/30 | Train Loss: 1.7578 | Val Loss: 2.9864 | Val Acc: 35.83%
Epoha 07/30 | Train Loss: 1.6652 | Val Loss: 1.4611 | Val Acc: 41.67%
Epoha 08/30 | Train Loss: 1.5880 | Val Loss: 2.1874 | Val Acc: 29.17%
Epoha 09/30 | Train Loss: 1.4670 | Val Loss: 4.1217 | Val Acc: 13.33%
Epoha 10/30 | Train Loss: 1.2345 | Val Loss: 3.0077 | Val Acc: 22.50%
Epoha 11/30 | Train Loss: 1.1096 | Val Loss: 1.8892 | Val Acc: 34.17%
Epoha 12/30 | Train Loss: 1.1107 | Val Loss: 2.1123 | Val Acc: 40.00%
Epoha 13/30 | Train Loss: 1.0183 | Val Loss: 2.6340 | Val Acc: 33.33%
Epoha 14/3

In [32]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_256CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_256N.pth", map_location=device))
model.eval()

sample_idx = 57
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka/s)")

=== Predikcija za 58. fajl u test skupu ===
Stvarna emocija (Ground Truth): surprised
Predviđena emocija:             surprised (98.99%)
Vreme pojedinačne inferencije:  203.845 ms

Verovatnoće po klasama:
  neutral   :   0.00%
  calm      :   0.00%
  happy     :   0.65%
  sad       :   0.00%
  angry     :   0.07%
  fearful   :   0.00%
  disgust   :   0.28%
  surprised :  98.99%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           9.9082 s
Prosečno vreme po uzorku:  82.568 ms
Brzina obrade (throughput): 12.11 FPS (uzoraka/s)


#### 512N

In [33]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "mfcc512N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_mfcc_512N_best.pth")


class MFCC_512CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(MFCC_512CNN_Small, self).__init__()
        n = 512
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

#m METRIKE 512N

def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = MFCC_512CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}


# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_mfcc:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_mfcc:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), best_model_path)
        saved_flag = " [Model je sačuvan]"

    
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_mfcc:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)

        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))


# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

#finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))




Koristi se uređaj: cpu
Epoha 01/30 | Train Loss: 2.9344 | Val Loss: 1.9912 | Val Acc: 19.17% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.4301 | Val Loss: 4.6601 | Val Acc: 16.67%
Epoha 03/30 | Train Loss: 2.1617 | Val Loss: 1.8895 | Val Acc: 33.33% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.0446 | Val Loss: 1.8823 | Val Acc: 31.67% [Model je sačuvan]
Epoha 05/30 | Train Loss: 1.9659 | Val Loss: 5.1150 | Val Acc: 17.50%
Epoha 06/30 | Train Loss: 2.0047 | Val Loss: 3.8674 | Val Acc: 17.50%
Epoha 07/30 | Train Loss: 1.8558 | Val Loss: 1.4813 | Val Acc: 43.33% [Model je sačuvan]
Epoha 08/30 | Train Loss: 1.7588 | Val Loss: 2.9054 | Val Acc: 16.67%
Epoha 09/30 | Train Loss: 1.7623 | Val Loss: 2.0397 | Val Acc: 31.67%
Epoha 10/30 | Train Loss: 1.6334 | Val Loss: 3.7349 | Val Acc: 25.83%
Epoha 11/30 | Train Loss: 1.6156 | Val Loss: 4.5255 | Val Acc: 20.83%
Epoha 12/30 | Train Loss: 1.4139 | Val Loss: 1.7624 | Val Acc: 45.00%
Epoha 13/30 | Train Loss: 1.2695 | Val Loss: 1.8047 | Val Acc

In [34]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MFCC_512CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_mfcc_512N.pth", map_location=device))
model.eval()

sample_idx = 7
x_sample, y_true_idx, _ = test_loader_mfcc.dataset[sample_idx]


inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100

print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_mfcc:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

=== Predikcija za 8. fajl u test skupu ===
Stvarna emocija (Ground Truth): fearful
Predviđena emocija:             disgust (93.60%)
Vreme pojedinačne inferencije:  510.974 ms

Verovatnoće po klasama:
  neutral   :   0.09%
  calm      :   0.59%
  happy     :   2.00%
  sad       :   2.92%
  angry     :   0.38%
  fearful   :   0.16%
  disgust   :  93.60%
  surprised :   0.27%




KeyboardInterrupt: 